In [25]:
import numpy as np
import pandas as pd
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score

In [26]:
file_path = 'Data/dane.csv'

heart_test = pd.read_csv('Data/heart_test.csv')
heart_train = pd.read_csv('Data/heart_train.csv')
diabetes_test = pd.read_csv('Data/diabetes_test.csv')
diabetes_train = pd.read_csv('Data/diabetes_train.csv')
cancer_test = pd.read_csv('Data/cancer_test.csv')
cancer_train = pd.read_csv('Data/cancer_train.csv')
alzheimer_test = pd.read_csv('Data/alzheimer_test.csv')
alzheimer_train = pd.read_csv('Data/alzheimer_train.csv')

datasets = {
    "heart": (heart_train, heart_test),
    "diabetes": (diabetes_train, diabetes_test),
    "cancer": (cancer_train, cancer_test),
    "alzheimer": (alzheimer_train, alzheimer_test)
}

In [27]:
n_random = 100
np.random.seed(42)

from scipy.stats import randint

param_dist_knn = {
    'n_neighbors': randint(1, 31),           # liczba sąsiadów w zakresie [1, 30]
    'weights': ['uniform', 'distance'],      # sposób ważenia sąsiadów
    'p': randint(1, 3),                      # 1 = Manhattan, 2 = Euklides
}



In [28]:
all_results = []

for name, (train, test) in datasets.items():
    print(f"Trenuję model KNN dla: {name}")

    X_train, y_train = train.iloc[:, :-1], train.iloc[:, -1]
    X_test, y_test = test.iloc[:, :-1], test.iloc[:, -1]

    model = KNeighborsClassifier()

    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_dist_knn,
        n_iter=100,
        scoring='roc_auc',
        cv=5,
        verbose=1,
        random_state=42,
        n_jobs=-1
    )

    search.fit(X_train, y_train)

    cv_results = pd.DataFrame(search.cv_results_)

    for i, params in enumerate(search.cv_results_['params']):
        model = KNeighborsClassifier(**params)
        model.fit(X_train, y_train)
        y_proba = model.predict_proba(X_test)[:, 1]
        test_auc = roc_auc_score(y_test, y_proba)
    
        # podstawowy rekord
        result = {
            "dataset": name,
            "cv_roc_auc": search.cv_results_['mean_test_score'][i],
            "test_roc_auc": test_auc
        }
    
        # dodaj osobno każdy parametr jako kolumnę
        for param_name, param_value in params.items():
            result[param_name] = param_value
    
        all_results.append(result)


results_df = pd.DataFrame(all_results)

Trenuję model KNN dla: heart
Fitting 5 folds for each of 100 candidates, totalling 500 fits
Trenuję model KNN dla: diabetes
Fitting 5 folds for each of 100 candidates, totalling 500 fits
Trenuję model KNN dla: cancer
Fitting 5 folds for each of 100 candidates, totalling 500 fits
Trenuję model KNN dla: alzheimer
Fitting 5 folds for each of 100 candidates, totalling 500 fits


In [29]:
best_per_dataset = (
    results_df
    .sort_values(by=["dataset", "test_roc_auc"], ascending=[True, False])
    .groupby("dataset", as_index=False)
    .first()
)

param_cols = [col for col in results_df.columns if col not in ["dataset", "cv_roc_auc", "test_roc_auc"]]
params_df = best_per_dataset[param_cols]

aggregated_params = {}

for col in params_df.columns:
    if col.lower() in ["n_neighbors"]:
        # Średnia i zaokrąglenie do najbliższej liczby całkowitej
        aggregated_params[col] = int(round(params_df[col].mean()))
    else:
        aggregated_params[col] = params_df[col].mode().iloc[0]


mean_params = pd.Series(aggregated_params)

print("Średnie najlepsze parametry:")
print(mean_params)


Średnie najlepsze parametry:
n_neighbors          24
p                     1
weights        distance
dtype: object


In [30]:
mean_results = []

print(f"Testuję wspólne średnie parametry: {mean_params.to_dict()}")

for name, (train, test) in datasets.items():
    print(f"\nTrenuję model KNN na średnich parametrach dla: {name}")

    X_train, y_train = train.iloc[:, :-1], train.iloc[:, -1]
    X_test, y_test = test.iloc[:, :-1], test.iloc[:, -1]

    model = KNeighborsClassifier(**mean_params.to_dict())
    model.fit(X_train, y_train)

    y_proba = model.predict_proba(X_test)[:, 1]
    mean_auc = roc_auc_score(y_test, y_proba)

    mean_results.append({
        "dataset": name,
        "star_test_roc_auc": mean_auc
    })

mean_df = pd.DataFrame(mean_results)

print("\nWyniki AUC dla wspólnych średnich parametrów na wszystkich zbiorach danych:")
print(mean_df)

Testuję wspólne średnie parametry: {'n_neighbors': 24, 'p': 1, 'weights': 'distance'}

Trenuję model KNN na średnich parametrach dla: heart

Trenuję model KNN na średnich parametrach dla: diabetes

Trenuję model KNN na średnich parametrach dla: cancer

Trenuję model KNN na średnich parametrach dla: alzheimer

Wyniki AUC dla wspólnych średnich parametrów na wszystkich zbiorach danych:
     dataset  star_test_roc_auc
0      heart           0.674140
1   diabetes           0.800597
2     cancer           0.772560
3  alzheimer           0.737553


In [31]:
results_df = results_df.merge(mean_df, on="dataset")
results_df["diff_from_mean"] = results_df["star_test_roc_auc"] - results_df["test_roc_auc"]
results_df

results_df = results_df.sort_values(by="diff_from_mean", ascending=True)

results_df = results_df[
    ["dataset", "n_neighbors", "p", "weights", "cv_roc_auc", "test_roc_auc", "star_test_roc_auc", "diff_from_mean"]
]

results_df.to_csv("Results/knn_random.csv", index=False)

In [32]:
results_df

,dataset,n_neighbors,p,weights,cv_roc_auc,test_roc_auc,star_test_roc_auc,diff_from_mean
393,alzheimer,30,1,distance,0.774266,0.753539,0.737553,-0.015986
383,alzheimer,30,1,distance,0.774266,0.753539,0.737553,-0.015986
317,alzheimer,30,1,distance,0.774266,0.753539,0.737553,-0.015986
369,alzheimer,30,1,uniform,0.772521,0.751732,0.737553,-0.014179
333,alzheimer,29,1,distance,0.774438,0.751724,0.737553,-0.014171
...,...,...,...,...,...,...,...,...
182,diabetes,1,1,uniform,0.645553,0.649731,0.800597,0.150866
355,alzheimer,1,2,uniform,0.593513,0.568527,0.737553,0.169026
347,alzheimer,1,2,distance,0.593513,0.568527,0.737553,0.169026
382,alzheimer,1,1,uniform,0.602902,0.530717,0.737553,0.206836


In [39]:
# Tworzymy wiersz z mean_params
mean_row = {
    "dataset": "STAR",
    "cv_roc_auc": 0,
    "test_roc_auc": 0,
    "n_neighbors": mean_params["n_neighbors"],
    "p": mean_params["p"],
    "weights": mean_params["weights"]
}

final_df = pd.concat([best_per_dataset, pd.DataFrame([mean_row])], ignore_index=True)


In [40]:
final_df

,dataset,cv_roc_auc,test_roc_auc,n_neighbors,p,weights
0,alzheimer,0.774266,0.753539,30,1,distance
1,cancer,0.782323,0.780356,30,2,distance
2,diabetes,0.806526,0.812537,27,1,uniform
3,heart,0.657624,0.686630,10,2,distance
4,STAR,0.000000,0.000000,24,1,distance


In [41]:
results_df.to_csv("Results/knn_random_summary.csv", index=False)